# 04 · Binary Search

Binary search finds a value in a **sorted** array by repeatedly halving the search range: compare with the middle element, then discard the half that cannot contain the target. **Time:** O(log n)

The code below is exactly my original code; I added explanations/comments and ⚠️ notes where I found issues.

**Topics:** basic search · lower/upper bound · search insert position · floor & ceil · first/last occurrence & count · rotated array (distinct values & duplicates) · minimum in a rotated array

## 1. Classic binary search (iterative)
`low`/`high` are the ends of the current range. If `nums[mid] == target` → found; if `nums[mid] < target` → go right (`low = mid + 1`); otherwise go left (`high = mid - 1`).

> ⚠️ **Note:** `mid` is calculated once **before** the loop, so it never changes inside the loop; the loop also keeps running after a match is printed and always prints `-1` at the end. `mid` should be recomputed inside the loop and the function should stop after finding the target.

In [ ]:
#binary search
nums=[2,4,6,7,9,11,18,19]
low=0
high=len(nums)-1
# middle index (only computed once here, see note)
mid=(low+high)//2
target=13
# keep searching while the range is not empty
while low<=high:
  if nums[mid]==target:
    print(mid)
  elif nums[mid]<target:
    # target is bigger → discard the left half
    low=mid+1
  else:
    # target is smaller → discard the right half
    high=mid -1
print(-1)

-1


## 2. Binary search (recursive)
Same idea: the "loop" is a function calling itself on the smaller range. **Base case:** `low > high` means not found. Uses the global `target`.

In [ ]:
def binary_search(nums,low,high):
  mid=(low +high)//2
  # empty range → not found
  if low>high:
    return -1
  # found
  if nums[mid]==target:
    return mid
  elif nums[mid]<target:
    # search the right half
    return binary_search(nums,mid+1,high)
  else:
    # search the left half
    return binary_search(nums,low,mid-1)
nums=[2,4,6,7,9,11,18,19]
low=0
high=len(nums)-1
target=18
print(binary_search(nums,low,high))

6


## 3. Lower bound
**Lower bound** = smallest index `i` with `nums[i] >= target`. When `nums[mid] >= target` we save `mid` as a candidate and keep searching to the **left** for an even smaller index. (`-1` means no such index.)

In [ ]:
#smallest index lower bound
n=len(nums)
# -1 = no element >= target
lower_bound=-1
low=0
high=n-1
while low<=high:
  mid =(low+high)//2
  # possible answer
  if nums[mid]>=target:
    # remember it and look further left
    lower_bound=mid
    high=mid-1
  else:
    low=mid+1
print(lower_bound)

6


## 4. Upper bound
**Upper bound** = smallest index `i` with `nums[i] > target` (strictly greater). The array length `n` is the default when no such element exists.

In [ ]:
#upper bound
nums=[1,1,1,1,1,2,3,3,5,6,7,7,7,7,9,12,12,13]
n=len(nums)
# default: insert at the end
upper_bound=n
low=0
high=n-1
target=1

while low<=high:
  mid=(low+high)//2
  # strictly greater → candidate
  if nums[mid]>target:
    upper_bound=mid
    high=mid-1
  else:
    low=mid+1
print(upper_bound)

5


## 5. Search insert position
Return the index of the target, or the index where it should be inserted. This is exactly the **lower bound**.

> ⚠️ **Note:** this linear-scan version prints several answers for one input, and `nums[i-1]` at `i = 0` reads the **last** element (negative index). Using the lower-bound function above avoids both problems.

In [ ]:
nums=[1,3,4,5,8,9,14,15,19,20,21]
n=len(nums)
target=3
for i in range(0,n):
  # found
  if nums[i]==target:
    print(i)
  # target fits between two elements
  if nums[i-1]<target<nums[i]:
    print(i)
  elif target>nums[n-1]:
    print(n)
  elif target<nums[0]:
    print(0)


1


## 6. Floor and ceil
**Floor** = largest value `<= target`; **ceil** = smallest value `>= target`. If `nums[mid] > target` it is a ceil candidate (go left); if it is smaller it is a floor candidate (go right). `-1` means it does not exist.

In [ ]:
def floor_ceil(nums,target):
    n=len(nums)
    floor=-1
    ceil=-1
    low=0
    high=len(nums)-1
    while low<=high:
      mid=(low+high)//2
      # exact match is both floor and ceil
      if nums[mid]==target:
        return [nums[mid],nums[mid]]
      elif nums[mid]>target:
        # bigger than target → ceil candidate
        ceil=nums[mid]
        high=mid-1
      else:
        # smaller than target → floor candidate
        floor=nums[mid]
        low=mid+1
    return ([floor,ceil])
nums=[1,3,4,5,8,9,14,15,19,20,21]
target=4
print(floor_ceil(nums,target))

[4, 4]


## 7. First and last occurrence
- **First occurrence** = lower bound (if `nums[lb] == target`)
- **Last occurrence** = upper bound `- 1`

So `searchRange` returns `[first, last]` in O(log n).

In [ ]:
class Solution():
  def lower_bound(self, nums, target):
    n=len(nums)
    lower_bound=-1
    low=0
    high=n-1
    while low<=high:
      mid =(low+high)//2
      if nums[mid]>=target:
        lower_bound=mid
        high=mid-1
      else:
        low=mid+1
    return (lower_bound)

  def upper_bound(self, nums, target):
    n=len(nums)
    upper_bound=n
    low=0
    high=n-1

    while low<=high:
      mid=(low+high)//2
      if nums[mid]>target:
        upper_bound=mid
        high=mid-1
      else:
        low=mid+1
    return (upper_bound)

  # returns [first index, last index] of target
  def searchRange(self, nums: list[int], target: int) -> list[int]:
    lb=self.lower_bound(nums, target)
    # target is not present
    if lb==-1 or lb >= len(nums) or nums[lb] != target:
      return [-1,-1]
    ub=self.upper_bound(nums, target)
    # last occurrence is just before the upper bound
    return [lb,ub-1]

nums=[1,1,3,4,4,4,4,5,7,8,8,8,9,9,9,10]
target=9
sol = Solution()
print(sol.searchRange(nums,target))

[12, 14]


## 8. Count occurrences
Linear scan counting matches, O(n). With binary search it is `upper_bound − lower_bound`, O(log n).

In [ ]:
#count the occurrence of a number
nums=[1,1,3,4,4,4,4,5,7,8,8,8,9,9,9,10]
n=len(nums)
count=0
target=3
for i in range(0,n):
  if nums[i]==target:
    # one more match
    count+=1
  else:
    pass
print(count)

1


## 9. Linear search (first index)
Scan from the left and print the first index where `nums[i] == target`. `found` records whether it was ever found.

In [ ]:
found=True
target=10
n=len(nums)
for i in range(0,n):
  # found at index i
  if nums[i]==target:
    print(i)
    found=True
    # stop at the first match
    break
  else:
    found=False
if found==False:
  print(-1)


15


## 10. Search in a rotated sorted array (distinct values)
The array is sorted but rotated, e.g. `[7,8,9,10,1,2,3,5,6]`. **At least one half around `mid` is always sorted.**
1. If `nums[mid] == target` → return `mid`.
2. Find the sorted half (`nums[mid] <= nums[high]` → right half is sorted, otherwise the left half).
3. If the target lies inside that sorted half, search it; otherwise search the other half.

In [ ]:
def search_rotated(nums,target):
    n=len(nums)
    low=0
    high=n-1
    while low<=high:
      mid=(low+high)//2
      if nums[mid]==target:
        return(mid)
      # right half [mid..high] is sorted
      if nums[mid]<=nums[high]:
        # target is inside the sorted right half
        if nums[mid]<=target<=nums[high]:
          low=mid+1
        else:
          high=mid-1
      else:
        # target is inside the sorted left half
        if nums[low]<=target<=nums[mid]:
          high=mid-1
        else:
          low=mid+1
    return(-1)
nums=[7,8,9,10,1,2,3,5,6]
target=8
print(search_rotated(nums,target))

1


The same function on an array **with duplicates** happens to work for this input, but it is not guaranteed in general:

In [ ]:
nums=[12,12,13,14,1,2,2,3,3,3,4,9,9,9,10,10,11,11]
target=9
print(search_rotated(nums,target))


13


## 11. Rotated array with duplicates
When `nums[low] == nums[mid] == nums[high]` we cannot tell which half is sorted (for example `[1,1,1,0,1]`). The only safe move is to shrink the range from both sides (`low += 1`, `high -= 1`). Worst case O(n).

In [ ]:
def search_rotated_duplicate(nums,target):
    n=len(nums)
    low=0
    high=n-1
    while low<=high:
      mid=(low+high)//2
      if nums[mid]==target:
        return True
      # cannot decide → trim both ends
      if nums[low]==nums[mid]==nums[high]:
        low+=1
        high-=1
      elif nums[mid]<=nums[high]:

        if nums[mid]<=target<=nums[high]:
            low=mid+1
        else:
            high=mid-1
      else:
          if nums[low]<=target<=nums[mid]:
            high=mid-1
          else:
            low=mid+1
    return False
nums=[7,7,7,7,7,7,7,1,7,7,7,7,7,7,7,7,7]
target=6
print(search_rotated_duplicate(nums,target))

False


## 12. Minimum of a rotated sorted array
**Brute force:** scan all elements.

> ⚠️ **Note:** `min=float("-inf")` should be `+inf` (otherwise the result is always `-inf`), and the variable `min` hides the built-in `min()` function, so `min(min,nums[i])` fails. This cell was never run.

In [ ]:
def mini_num_sorted_array(nums:list[int])->int:
  n=len(nums)
  # should start at +infinity (see note)
  min=float("-inf")
  for i in range(0,n):
    min=min(min,nums[i])
  return min

**Binary search:** in each step one half is sorted and its smallest value is at its left end. Record that candidate, then discard that half. **Time:** O(log n)

In [ ]:
def min_num_sorted_array_binary_search(nums:list[int])->int:
  n=len(nums)
  low=0
  high=n-1
  mini=float("inf")
  while low<=high:
    mid=(low+high)//2
    # right half sorted → its minimum is nums[mid]
    if nums[mid]<=nums[high]:
      # candidate from the right half
      mini=min(mini,nums[mid])
      high=mid-1
    else:
      # left half sorted → its minimum is nums[low]
      mini=min(mini,nums[low])
      low=mid+1
  return mini
nums=[7,8,9,1,2,3,4]
print(min_num_sorted_array_binary_search(nums))


1


---
### ✅ Key takeaways
- Binary search needs a **sorted** (or monotonic) search space.
- **Lower/upper bound** solve insert position, first/last occurrence and counting.
- **Rotated array:** one half is always sorted → test whether the target lies in it.
- With **duplicates**, handle `nums[low] == nums[mid] == nums[high]` by shrinking both ends.